# 07. Full Model — 全要素統合 (2026-03-08)

## Recruit Restaurant Visitor Forecasting

**目的**: 06_model_designの設計に基づき、全改善要素を統合した最終モデルを構築する。

### 改善要素コード表

| コード | 要素 | 説明 |
|--------|------|------|
| `li` | Lag Imputation | テスト期間のlag/rolling NaN修正（実データ活用 + store×DOW補完） |
| `gw` | Golden Week | GW専用特徴量・特殊期間の精密化 |
| `wp` | Weather Proxy | 気温・梅雨の季節代理変数 |
| `op` | Optuna All | 3モデル全てをOptuna最適化 |
| `st` | Stacking | 2段階アンサンブル（Level2: Ridge） |

### モデル命名規則

```
{algorithm}_{elements}_{version}
例: lgbm_li-gw-wp-op_v1 = LightGBM + lag補完 + GW対応 + 天気proxy + Optuna
```

### Notebook Flow
1. データ読み込み & 前処理
2. 拡張特徴量パイプライン（li + gw + wp）
3. 5-fold TimeSeriesSplit CV
4. Optuna全モデル最適化（op）
5. スタッキング（st）
6. 最終学習 → Submission生成
7. モデルレジストリ & 要素別寄与分析の準備

**評価指標**: RMSLE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
import time
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

%matplotlib inline
plt.rcParams['font.family'] = 'IPAGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 12
plt.rcParams['figure.facecolor'] = 'white'

INPUT_DIR = Path('../input')
OUTPUT_DIR = Path('../output')
MODEL_DIR = Path('../models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

def rmsle(y_true, y_pred):
    """RMSLE = RMSE in log1p space."""
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(np.maximum(y_pred, 0))))

print('Setup complete.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. データ読み込み
# ══════════════════════════════════════════════════════════════

air_visit = pd.read_csv(INPUT_DIR / 'air_visit_data.csv', parse_dates=['visit_date'])
air_store = pd.read_csv(INPUT_DIR / 'air_store_info.csv')
air_reserve = pd.read_csv(INPUT_DIR / 'air_reserve.csv', parse_dates=['visit_datetime', 'reserve_datetime'])
hpg_reserve = pd.read_csv(INPUT_DIR / 'hpg_reserve.csv', parse_dates=['visit_datetime', 'reserve_datetime'])
store_relation = pd.read_csv(INPUT_DIR / 'store_id_relation.csv')
date_info = pd.read_csv(INPUT_DIR / 'date_info.csv', parse_dates=['calendar_date'])
submission = pd.read_csv(INPUT_DIR / 'sample_submission.csv')

# Parse submission
submission['air_store_id'] = submission['id'].apply(lambda x: '_'.join(x.split('_')[:-1]))
submission['visit_date'] = pd.to_datetime(submission['id'].apply(lambda x: x.split('_')[-1]))

# Basic derived columns
air_visit['dow'] = air_visit['visit_date'].dt.dayofweek
air_visit['log_visitors'] = np.log1p(air_visit['visitors'])

# Validation folds (from EDA Section 11: 39-day windows matching test period)
VAL_FOLDS = [
    ('2016-04-23', '2016-05-31'),  # Fold 1: GW ★最重要
    ('2016-07-16', '2016-08-23'),  # Fold 2: 夏
    ('2016-10-15', '2016-11-22'),  # Fold 3: 秋
    ('2016-12-16', '2017-01-23'),  # Fold 4: 年末年始
    ('2017-03-15', '2017-04-22'),  # Fold 5: テスト直前
]

print(f'air_visit: {air_visit.shape}')
print(f'Submission: {submission.shape} ({submission["visit_date"].min().date()} ~ {submission["visit_date"].max().date()})')
print(f'Test stores: {submission["air_store_id"].nunique()}')

---
## 2. 拡張特徴量パイプライン

### 06_model_designからの改善点

| 改善 | 06の問題 | 07の対策 | 期待効果 |
|------|---------|---------|---------|
| **[li] Lag修正** | テストのlag_7〜35が100% NaN | train→test方向のlag計算 + store×DOW補完 | テスト精度の大幅改善 |
| **[gw] GW対応** | special_period=GWのみ | GW日数カウント・GW内位置・連休構造 | GW期間のFold1改善 |
| **[wp] 天気proxy** | なし | 月×緯度→気温proxy、梅雨フラグ | 季節効果の補完 |

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2a. 日付特徴量（祝日・特殊期間）[gw] 拡張版
# ══════════════════════════════════════════════════════════════

date_feat = date_info.rename(columns={'calendar_date': 'visit_date'}).copy()
date_feat = date_feat.sort_values('visit_date').reset_index(drop=True)

# テスト期間をカバーするよう拡張（date_infoにない日付があれば追加）
test_dates = pd.date_range('2017-04-23', '2017-05-31')
existing_dates = set(date_feat['visit_date'])
new_rows = []
for d in test_dates:
    if d not in existing_dates:
        new_rows.append({'visit_date': d, 'day_of_week': d.strftime('%A'), 'holiday_flg': 0})
if new_rows:
    date_feat = pd.concat([date_feat, pd.DataFrame(new_rows)], ignore_index=True)
    date_feat = date_feat.sort_values('visit_date').reset_index(drop=True)

# 祝日前日・翌日（EDA Sec 7c）
date_feat['holiday_tomorrow'] = date_feat['holiday_flg'].shift(-1).fillna(0).astype(int)
date_feat['holiday_yesterday'] = date_feat['holiday_flg'].shift(1).fillna(0).astype(int)
date_feat['is_before_holiday'] = ((date_feat['holiday_flg'] == 0) & (date_feat['holiday_tomorrow'] == 1)).astype(int)
date_feat['is_after_holiday'] = ((date_feat['holiday_flg'] == 0) & (date_feat['holiday_yesterday'] == 1)).astype(int)

# [gw] 特殊期間の精密化
def get_special_period(date):
    m, d = date.month, date.day
    if m == 1 and d <= 3: return 'new_year'
    if m == 4 and d >= 29: return 'golden_week'
    if m == 5 and d <= 7: return 'golden_week'
    if m == 8 and 13 <= d <= 16: return 'obon'
    if m == 12 and d >= 25: return 'year_end'
    if m == 3 and 25 <= d <= 31: return 'spring_graduation'  # 送別会シーズン
    return 'normal'

date_feat['special_period'] = date_feat['visit_date'].apply(get_special_period)

# [gw] GW内の位置（0=GW外、1=GW初日、2=2日目...）
def gw_position(date):
    m, d = date.month, date.day
    if m == 4 and d >= 29:
        return d - 28  # 4/29→1, 4/30→2
    elif m == 5 and d <= 7:
        return d + 2   # 5/1→3, 5/2→4, ..., 5/7→9
    return 0

date_feat['gw_position'] = date_feat['visit_date'].apply(gw_position)

# [gw] 連休の長さ（連続する祝日+土日のブロック長）
# 週末と祝日を統合した「実質的な休日」で連休を計算
date_feat['dow_num'] = date_feat['visit_date'].dt.dayofweek
date_feat['is_weekend'] = (date_feat['dow_num'] >= 5).astype(int)
date_feat['is_off'] = ((date_feat['holiday_flg'] == 1) | (date_feat['is_weekend'] == 1)).astype(int)

# 連続休日ブロックの長さ
off_blocks = (date_feat['is_off'] != date_feat['is_off'].shift()).cumsum()
block_sizes = date_feat.groupby(off_blocks)['is_off'].transform('sum')
date_feat['holiday_streak'] = (block_sizes * date_feat['is_off']).astype(int)

# [gw] 連休内の位置（連休の何日目か）
date_feat['streak_position'] = 0
for block_id in date_feat[date_feat['is_off'] == 1].groupby(off_blocks).groups:
    mask = off_blocks == block_id
    if date_feat.loc[mask, 'is_off'].sum() > 0:
        positions = range(1, mask.sum() + 1)
        date_feat.loc[mask, 'streak_position'] = list(positions)

# 不要列の削除
date_feat = date_feat.drop(columns=['dow_num', 'is_weekend', 'is_off'])

print('Date features created:')
print(f'  Columns: {[c for c in date_feat.columns if c != "day_of_week"]}')
print(f'  Special periods: {date_feat["special_period"].value_counts().to_dict()}')
print(f'  GW position max: {date_feat["gw_position"].max()}')
print(f'  Holiday streak max: {date_feat["holiday_streak"].max()}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2b. 予約データ集約（EDA Sec 9）
# ══════════════════════════════════════════════════════════════

air_reserve['visit_date'] = air_reserve['visit_datetime'].dt.floor('D')
air_res_agg = air_reserve.groupby(['air_store_id', 'visit_date']).agg(
    air_res_visitors=('reserve_visitors', 'sum'),
    air_res_count=('reserve_visitors', 'count'),
).reset_index()

hpg_reserve['visit_date'] = hpg_reserve['visit_datetime'].dt.floor('D')
hpg_res_mapped = hpg_reserve.merge(store_relation, on='hpg_store_id', how='inner')
hpg_res_agg = hpg_res_mapped.groupby(['air_store_id', 'visit_date']).agg(
    hpg_res_visitors=('reserve_visitors', 'sum'),
    hpg_res_count=('reserve_visitors', 'count'),
).reset_index()

print(f'Air reservations: {air_res_agg.shape}, HPG reservations: {hpg_res_agg.shape}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2c. [wp] 天気Proxy特徴量
# ══════════════════════════════════════════════════════════════
# 実際の天気データは外部ソースが必要（11th place solutionで有効性確認済み）
# ここでは月×緯度から気温の季節代理変数を作成

# 日本の月別平均気温の近似モデル（緯度ベース）
# 基準: 東京(35.7°N) → 年平均16°C, 振幅12°C
# 緯度1度あたり約-0.8°C
def temp_proxy(month, latitude, base_lat=35.7, base_temp=16.0, amplitude=12.0, lat_coeff=-0.8):
    """月と緯度から気温を推定する季節代理変数"""
    seasonal = amplitude * np.cos(2 * np.pi * (month - 8) / 12)  # 8月ピーク
    lat_effect = lat_coeff * (latitude - base_lat)
    return base_temp + seasonal + lat_effect

# 梅雨フラグ（6月上旬〜7月中旬）
def is_rainy_season(month, day):
    if month == 6: return 1
    if month == 7 and day <= 20: return 1
    return 0

# 季節カテゴリ
def get_season(month):
    if month in [3, 4, 5]: return 'spring'
    if month in [6, 7, 8]: return 'summer'
    if month in [9, 10, 11]: return 'autumn'
    return 'winter'

print('[wp] Weather proxy functions defined.')
print('  - temp_proxy(month, latitude): 月×緯度→推定気温')
print('  - is_rainy_season(month, day): 梅雨フラグ')
print('  - get_season(month): 四季カテゴリ')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2d. [li] 拡張特徴量生成関数（lag/rolling NaN修正版）
# ══════════════════════════════════════════════════════════════
#
# 06の問題: lag特徴量がtrain内のpivotからしか計算されず、テスト期間で100% NaN
# 修正: train+test期間を統合したpivotでlag計算 → trainの実データがtestのlagとして使える
#       残りのNaN → store×DOW median で補完 + is_lag_imputed フラグ

def build_features_full(train_df, target_dates_df, air_store_info, date_features,
                        air_res, hpg_res, compute_target=True):
    """
    全改善要素[li][gw][wp]を統合した特徴量パイプライン。
    
    [li] train→testにまたがるlag計算 + NaN補完
    [gw] GW詳細特徴量（位置・連休構造）
    [wp] 天気proxy（気温・梅雨・季節）
    """
    df = target_dates_df[['air_store_id', 'visit_date']].copy()
    
    # ── 時間特徴量 ──
    df['dow'] = df['visit_date'].dt.dayofweek
    df['month'] = df['visit_date'].dt.month
    df['day'] = df['visit_date'].dt.day
    df['week_of_year'] = df['visit_date'].dt.isocalendar().week.astype(int)
    
    # ── 日付特徴量マージ [gw] 拡張版 ──
    date_cols = ['visit_date', 'holiday_flg', 'is_before_holiday', 'is_after_holiday',
                 'special_period', 'holiday_streak', 'gw_position', 'streak_position']
    df = df.merge(date_features[date_cols], on='visit_date', how='left')
    
    # ── 店舗マスタ [wp] 緯度経度を天気proxyに活用 ──
    df = df.merge(air_store_info[['air_store_id', 'air_genre_name', 'air_area_name',
                                   'latitude', 'longitude']], on='air_store_id', how='left')
    df['prefecture'] = df['air_area_name'].apply(lambda x: x.split(' ')[0] if pd.notna(x) else 'unknown')
    
    # ── [wp] 天気proxy特徴量 ──
    df['temp_proxy'] = df.apply(lambda r: temp_proxy(r['month'], r['latitude']) 
                                 if pd.notna(r['latitude']) else np.nan, axis=1)
    df['is_rainy_season'] = df.apply(lambda r: is_rainy_season(r['month'], r['day']), axis=1)
    df['season'] = df['month'].apply(get_season)
    
    # ── 店舗レベル統計量 ──
    store_stats = train_df.groupby('air_store_id')['visitors'].agg(
        store_mean='mean', store_median='median', store_std='std',
        store_min='min', store_max='max', store_count='count'
    )
    df = df.merge(store_stats, on='air_store_id', how='left')
    
    # ── 店舗×曜日 統計量 ──
    store_dow_stats = train_df.groupby(['air_store_id', 'dow'])['visitors'].agg(
        store_dow_mean='mean', store_dow_median='median', store_dow_std='std',
        store_dow_count='count'
    )
    df = df.merge(store_dow_stats, on=['air_store_id', 'dow'], how='left')
    
    # ── ジャンル×曜日 ──
    train_genre = train_df.merge(air_store_info[['air_store_id', 'air_genre_name']], on='air_store_id')
    genre_dow_stats = train_genre.groupby(['air_genre_name', 'dow'])['visitors'].agg(
        genre_dow_mean='mean', genre_dow_median='median'
    )
    df = df.merge(genre_dow_stats, on=['air_genre_name', 'dow'], how='left')
    
    # ── [gw] ジャンル×特殊期間 統計量 ──
    train_with_sp = train_df.merge(date_features[['visit_date', 'special_period']], on='visit_date')
    train_with_sp = train_with_sp.merge(air_store_info[['air_store_id', 'air_genre_name']], on='air_store_id')
    genre_sp = train_with_sp.groupby(['air_genre_name', 'special_period'])['visitors'].agg(
        genre_sp_mean='mean', genre_sp_median='median'
    )
    df = df.merge(genre_sp, on=['air_genre_name', 'special_period'], how='left')
    
    # ══════════════════════════════════════════════════════════
    # [li] Lag特徴量: train→test方向の修正版
    # ══════════════════════════════════════════════════════════
    # train+targetの全日付×全店舗のpivotを作成し、lagを計算
    all_dates = sorted(set(train_df['visit_date'].unique()) | set(target_dates_df['visit_date'].unique()))
    all_stores = train_df['air_store_id'].unique()
    
    # trainデータのvisitors pivotを作成（target期間はNaN）
    visit_pivot = train_df.pivot_table(index='visit_date', columns='air_store_id',
                                        values='visitors', aggfunc='first')
    # 全日付にreindex（target期間の行が追加される）
    visit_pivot = visit_pivot.reindex(pd.DatetimeIndex(all_dates))
    
    for lag in [7, 14, 21, 28, 35]:
        lag_vals = visit_pivot.shift(lag).stack().reset_index()
        lag_vals.columns = ['visit_date', 'air_store_id', f'lag_{lag}']
        df = df.merge(lag_vals, on=['air_store_id', 'visit_date'], how='left')
    
    # [li] Rolling統計量（同様にtrain→testに拡張）
    for window in [7, 14, 28]:
        roll_mean = visit_pivot.rolling(window, min_periods=1).mean()
        # shift by 1 to prevent leakage
        roll_shifted = roll_mean.shift(1).stack().reset_index()
        roll_shifted.columns = ['visit_date', 'air_store_id', f'rolling_mean_{window}']
        df = df.merge(roll_shifted, on=['air_store_id', 'visit_date'], how='left')
    
    # [li] NaN補完: store×DOW median で埋める + フラグ
    lag_cols = [f'lag_{l}' for l in [7, 14, 21, 28, 35]]
    roll_cols = [f'rolling_mean_{w}' for w in [7, 14, 28]]
    
    for col in lag_cols + roll_cols:
        df[f'{col}_imputed'] = df[col].isna().astype(int)
        # store×DOW median で補完
        if col in lag_cols:
            df[col] = df[col].fillna(df['store_dow_median'])
        else:
            df[col] = df[col].fillna(df['store_median'])
    
    # ── 予約特徴量 ──
    df = df.merge(air_res, on=['air_store_id', 'visit_date'], how='left')
    df = df.merge(hpg_res, on=['air_store_id', 'visit_date'], how='left')
    df['total_res_visitors'] = df['air_res_visitors'].fillna(0) + df['hpg_res_visitors'].fillna(0)
    df['total_res_count'] = df['air_res_count'].fillna(0) + df['hpg_res_count'].fillna(0)
    df['has_reservation'] = (df['total_res_visitors'] > 0).astype(int)
    
    # ── Target ──
    if compute_target and 'visitors' in target_dates_df.columns:
        df = df.merge(target_dates_df[['air_store_id', 'visit_date', 'visitors']],
                      on=['air_store_id', 'visit_date'], how='left')
    
    return df

print('build_features_full() defined with [li][gw][wp] improvements.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2e. 特徴量カラム定義
# ══════════════════════════════════════════════════════════════

CATEGORICAL_FEATURES = [
    'air_genre_name', 'air_area_name', 'prefecture',
    'special_period', 'season',  # [gw][wp]
]

NUMERIC_FEATURES = [
    # 時間
    'dow', 'month', 'day', 'week_of_year',
    # 祝日 [gw拡張]
    'holiday_flg', 'is_before_holiday', 'is_after_holiday',
    'holiday_streak', 'gw_position', 'streak_position',
    # 店舗統計量
    'store_mean', 'store_median', 'store_std', 'store_min', 'store_max', 'store_count',
    # 店舗×曜日
    'store_dow_mean', 'store_dow_median', 'store_dow_std', 'store_dow_count',
    # ジャンル×曜日
    'genre_dow_mean', 'genre_dow_median',
    # [gw] ジャンル×特殊期間
    'genre_sp_mean', 'genre_sp_median',
    # ラグ [li修正版]
    'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_35',
    # ローリング [li修正版]
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28',
    # [li] 補完フラグ
    'lag_7_imputed', 'lag_14_imputed', 'lag_21_imputed', 'lag_28_imputed', 'lag_35_imputed',
    'rolling_mean_7_imputed', 'rolling_mean_14_imputed', 'rolling_mean_28_imputed',
    # 予約
    'air_res_visitors', 'air_res_count', 'hpg_res_visitors', 'hpg_res_count',
    'total_res_visitors', 'total_res_count', 'has_reservation',
    # 位置
    'latitude', 'longitude',
    # [wp] 天気proxy
    'temp_proxy', 'is_rainy_season',
]

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
print(f'Total features: {len(ALL_FEATURES)} ({len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical)')
print(f'  [li] lag/rolling imputed flags: 8')
print(f'  [gw] GW features: gw_position, streak_position, genre_sp_mean/median')
print(f'  [wp] Weather proxy: temp_proxy, is_rainy_season, season')

---
## 3. ベースライン & CVフレームワーク

06_model_designのベースライン（store×DOW中央値: RMSLE 0.5908）を再現し、
拡張特徴量の効果を測定するための基準点とする。

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3a. ベースライン再現 + カテゴリカルエンコーダ準備
# ══════════════════════════════════════════════════════════════

# ── ベースライン: store×DOW median（06_model_designと同一） ──
baseline_scores = []
for val_start, val_end in VAL_FOLDS:
    val_start_ts, val_end_ts = pd.Timestamp(val_start), pd.Timestamp(val_end)
    train = air_visit[air_visit['visit_date'] < val_start_ts]
    val = air_visit[(air_visit['visit_date'] >= val_start_ts) & (air_visit['visit_date'] <= val_end_ts)].copy()
    store_dow_med = train.groupby(['air_store_id', 'dow'])['visitors'].median()
    store_med = train.groupby('air_store_id')['visitors'].median()
    global_med = train['visitors'].median()
    val['pred'] = val.apply(
        lambda r: store_dow_med.get((r['air_store_id'], r['dow']),
                  store_med.get(r['air_store_id'], global_med)), axis=1)
    baseline_scores.append(rmsle(val['visitors'], val['pred']))

mean_baseline = np.mean(baseline_scores)
print(f'Baseline (store×DOW median): Mean RMSLE = {mean_baseline:.4f} (±{np.std(baseline_scores):.4f})')

# ── カテゴリカルエンコーダ（XGBoost用） ──
label_encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    if col == 'air_genre_name':
        vals = list(air_store['air_genre_name'].dropna().unique())
    elif col == 'air_area_name':
        vals = list(air_store['air_area_name'].dropna().unique())
    elif col == 'prefecture':
        vals = list(air_store['air_area_name'].apply(lambda x: x.split(' ')[0]).unique())
    elif col == 'special_period':
        vals = ['normal', 'new_year', 'golden_week', 'obon', 'year_end', 'spring_graduation']
    elif col == 'season':
        vals = ['spring', 'summer', 'autumn', 'winter']
    else:
        vals = []
    le.fit(vals + ['unknown'])
    label_encoders[col] = le

print(f'Label encoders ready for {len(label_encoders)} categorical features.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3b. CVユーティリティ関数
# ══════════════════════════════════════════════════════════════

def encode_categoricals(df, cat_features, label_encoders):
    """Label-encode for XGBoost."""
    df = df.copy()
    for col in cat_features:
        if col in df.columns:
            df[col] = df[col].fillna('unknown').astype(str)
            df[col] = df[col].apply(lambda x: x if x in label_encoders[col].classes_ else 'unknown')
            df[col] = label_encoders[col].transform(df[col])
    return df

def unify_categories(X_train, X_val, cat_features):
    """Ensure train/val have same category levels for LightGBM."""
    X_tr, X_v = X_train.copy(), X_val.copy()
    for col in cat_features:
        X_tr[col] = X_tr[col].fillna('unknown').astype(str)
        X_v[col] = X_v[col].fillna('unknown').astype(str)
        all_cats = sorted(set(X_tr[col].unique()) | set(X_v[col].unique()))
        cat_type = pd.CategoricalDtype(categories=all_cats)
        X_tr[col] = X_tr[col].astype(cat_type)
        X_v[col] = X_v[col].astype(cat_type)
    return X_tr, X_v

def run_cv_full(model_name, train_fn, predict_fn, air_visit_df, val_folds,
                features, cat_features, collect_oof=False):
    """
    5-fold CV with build_features_full.
    collect_oof=True: OOF予測を収集（スタッキング用）
    """
    fold_scores = []
    oof_list = []
    total_time = 0
    
    for fold_idx, (val_start, val_end) in enumerate(val_folds):
        val_start_ts, val_end_ts = pd.Timestamp(val_start), pd.Timestamp(val_end)
        train_data = air_visit_df[air_visit_df['visit_date'] < val_start_ts].copy()
        val_data = air_visit_df[(air_visit_df['visit_date'] >= val_start_ts) & 
                                (air_visit_df['visit_date'] <= val_end_ts)].copy()
        if len(val_data) == 0:
            continue
        
        train_feat = build_features_full(train_data, train_data, air_store, date_feat,
                                          air_res_agg, hpg_res_agg, compute_target=True)
        val_feat = build_features_full(train_data, val_data, air_store, date_feat,
                                        air_res_agg, hpg_res_agg, compute_target=True)
        train_feat = train_feat.dropna(subset=['visitors'])
        val_feat = val_feat.dropna(subset=['visitors'])
        
        X_train = train_feat[features].copy()
        y_train = np.log1p(train_feat['visitors'])
        X_val = val_feat[features].copy()
        y_val = val_feat['visitors']
        
        t0 = time.time()
        model = train_fn(X_train, y_train, X_val, np.log1p(y_val), cat_features)
        elapsed = time.time() - t0
        total_time += elapsed
        
        y_pred_log = predict_fn(model, X_val, X_train, cat_features)
        y_pred = np.expm1(y_pred_log)
        y_pred = np.maximum(y_pred, 0)
        
        score = rmsle(y_val.values, y_pred)
        fold_scores.append(score)
        
        if collect_oof:
            oof_list.append(pd.DataFrame({
                'air_store_id': val_feat['air_store_id'].values,
                'visit_date': val_feat['visit_date'].values,
                'fold': fold_idx,
                'actual': y_val.values,
                f'pred_{model_name}': y_pred,
                f'pred_log_{model_name}': y_pred_log,
            }))
        
        print(f'  Fold {fold_idx+1}: {score:.4f} ({elapsed:.1f}s)')
    
    mean_score = np.mean(fold_scores)
    print(f'  Mean: {mean_score:.4f} (±{np.std(fold_scores):.4f}) | Total: {total_time:.1f}s')
    
    oof_df = pd.concat(oof_list, ignore_index=True) if collect_oof else None
    return fold_scores, total_time, oof_df

print('CV framework ready.')

---
## 4. [op] Optuna全モデル最適化

06_model_designではLightGBMのみOptuna調整 → 今回は3モデル全てを最適化。
各モデル30 trialsで探索し、最適パラメータでOOF予測を収集（スタッキング用）。

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4a. [op] Optuna — LightGBM
# ══════════════════════════════════════════════════════════════

def optuna_lgbm_objective(trial):
    params = {
        'objective': 'regression', 'metric': 'rmse', 'verbose': -1, 'seed': SEED,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
    }
    
    fold_scores = []
    for val_start, val_end in VAL_FOLDS:
        vs, ve = pd.Timestamp(val_start), pd.Timestamp(val_end)
        train_data = air_visit[air_visit['visit_date'] < vs]
        val_data = air_visit[(air_visit['visit_date'] >= vs) & (air_visit['visit_date'] <= ve)]
        
        train_feat = build_features_full(train_data, train_data, air_store, date_feat,
                                          air_res_agg, hpg_res_agg, True)
        val_feat = build_features_full(train_data, val_data, air_store, date_feat,
                                        air_res_agg, hpg_res_agg, True)
        train_feat = train_feat.dropna(subset=['visitors'])
        val_feat = val_feat.dropna(subset=['visitors'])
        
        X_tr, X_v = train_feat[ALL_FEATURES].copy(), val_feat[ALL_FEATURES].copy()
        X_tr, X_v = unify_categories(X_tr, X_v, CATEGORICAL_FEATURES)
        
        ts = lgb.Dataset(X_tr, np.log1p(train_feat['visitors']), categorical_feature=CATEGORICAL_FEATURES)
        vs_ds = lgb.Dataset(X_v, np.log1p(val_feat['visitors']), categorical_feature=CATEGORICAL_FEATURES, reference=ts)
        
        model = lgb.train(params, ts, num_boost_round=1000, valid_sets=[vs_ds],
                          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        
        y_pred = np.expm1(model.predict(X_v))
        fold_scores.append(rmsle(val_feat['visitors'].values, np.maximum(y_pred, 0)))
    
    return np.mean(fold_scores)

print('Running Optuna for LightGBM (30 trials)...')
study_lgbm = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgbm.optimize(optuna_lgbm_objective, n_trials=30)
best_lgbm_params = {**study_lgbm.best_params, 'objective': 'regression', 'metric': 'rmse', 'verbose': -1, 'seed': SEED}
print(f'\nBest LightGBM RMSLE: {study_lgbm.best_value:.4f}')
print(f'Best params: { {k: round(v, 4) if isinstance(v, float) else v for k, v in study_lgbm.best_params.items()} }')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4b. [op] Optuna — XGBoost
# ══════════════════════════════════════════════════════════════

def optuna_xgb_objective(trial):
    params = {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'verbosity': 0, 'seed': SEED,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'lambda': trial.suggest_float('lambda', 0.1, 10.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-3, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-3, 5.0, log=True),
    }
    
    fold_scores = []
    for val_start, val_end in VAL_FOLDS:
        vs, ve = pd.Timestamp(val_start), pd.Timestamp(val_end)
        train_data = air_visit[air_visit['visit_date'] < vs]
        val_data = air_visit[(air_visit['visit_date'] >= vs) & (air_visit['visit_date'] <= ve)]
        
        train_feat = build_features_full(train_data, train_data, air_store, date_feat,
                                          air_res_agg, hpg_res_agg, True)
        val_feat = build_features_full(train_data, val_data, air_store, date_feat,
                                        air_res_agg, hpg_res_agg, True)
        train_feat = train_feat.dropna(subset=['visitors'])
        val_feat = val_feat.dropna(subset=['visitors'])
        
        X_tr = encode_categoricals(train_feat[ALL_FEATURES], CATEGORICAL_FEATURES, label_encoders)
        X_v = encode_categoricals(val_feat[ALL_FEATURES], CATEGORICAL_FEATURES, label_encoders)
        
        dtrain = xgb.DMatrix(X_tr, np.log1p(train_feat['visitors']))
        dval = xgb.DMatrix(X_v, np.log1p(val_feat['visitors']))
        
        model = xgb.train(params, dtrain, num_boost_round=1000,
                          evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
        
        y_pred = np.expm1(model.predict(dval))
        fold_scores.append(rmsle(val_feat['visitors'].values, np.maximum(y_pred, 0)))
    
    return np.mean(fold_scores)

print('Running Optuna for XGBoost (30 trials)...')
study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(optuna_xgb_objective, n_trials=30)
best_xgb_params = {**study_xgb.best_params, 'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'verbosity': 0, 'seed': SEED}
print(f'\nBest XGBoost RMSLE: {study_xgb.best_value:.4f}')
print(f'Best params: { {k: round(v, 4) if isinstance(v, float) else v for k, v in study_xgb.best_params.items()} }')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4c. [op] Optuna — CatBoost
# ══════════════════════════════════════════════════════════════

def optuna_cb_objective(trial):
    cb_params = {
        'iterations': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 10.0, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 50),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_seed': SEED, 'verbose': 0, 'early_stopping_rounds': 50,
    }
    
    fold_scores = []
    for val_start, val_end in VAL_FOLDS:
        vs, ve = pd.Timestamp(val_start), pd.Timestamp(val_end)
        train_data = air_visit[air_visit['visit_date'] < vs]
        val_data = air_visit[(air_visit['visit_date'] >= vs) & (air_visit['visit_date'] <= ve)]
        
        train_feat = build_features_full(train_data, train_data, air_store, date_feat,
                                          air_res_agg, hpg_res_agg, True)
        val_feat = build_features_full(train_data, val_data, air_store, date_feat,
                                        air_res_agg, hpg_res_agg, True)
        train_feat = train_feat.dropna(subset=['visitors'])
        val_feat = val_feat.dropna(subset=['visitors'])
        
        X_tr, X_v = train_feat[ALL_FEATURES].copy(), val_feat[ALL_FEATURES].copy()
        for col in CATEGORICAL_FEATURES:
            X_tr[col] = X_tr[col].fillna('unknown').astype(str)
            X_v[col] = X_v[col].fillna('unknown').astype(str)
        cat_indices = [list(X_tr.columns).index(c) for c in CATEGORICAL_FEATURES]
        
        model = CatBoostRegressor(**cb_params, cat_features=cat_indices)
        model.fit(X_tr, np.log1p(train_feat['visitors']),
                  eval_set=(X_v, np.log1p(val_feat['visitors'])), verbose=0)
        
        y_pred = np.expm1(model.predict(X_v))
        fold_scores.append(rmsle(val_feat['visitors'].values, np.maximum(y_pred, 0)))
    
    return np.mean(fold_scores)

print('Running Optuna for CatBoost (30 trials)...')
study_cb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_cb.optimize(optuna_cb_objective, n_trials=30)
best_cb_params = {k: v for k, v in study_cb.best_params.items()}
print(f'\nBest CatBoost RMSLE: {study_cb.best_value:.4f}')
print(f'Best params: { {k: round(v, 4) if isinstance(v, float) else v for k, v in study_cb.best_params.items()} }')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4d. Optunaチューニング結果サマリー & OOF予測収集
# ══════════════════════════════════════════════════════════════

print('=' * 70)
print('OPTUNA TUNING SUMMARY')
print('=' * 70)
print(f'  LightGBM:  {study_lgbm.best_value:.4f}')
print(f'  XGBoost:   {study_xgb.best_value:.4f}')
print(f'  CatBoost:  {study_cb.best_value:.4f}')
print(f'  Baseline:  {mean_baseline:.4f}')

# ── OOF予測を収集（スタッキング用） ──
# 最適パラメータでCV再実行し、fold別のOOF予測を取得

# LightGBM train/predict functions
def train_lgbm_tuned(X_tr, y_tr, X_v, y_v, cat_feats):
    X_tr, X_v = unify_categories(X_tr, X_v, cat_feats)
    ts = lgb.Dataset(X_tr, y_tr, categorical_feature=cat_feats)
    vs = lgb.Dataset(X_v, y_v, categorical_feature=cat_feats, reference=ts)
    model = lgb.train(best_lgbm_params, ts, num_boost_round=1000, valid_sets=[vs],
                      callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    model._X_tr_cats = X_tr  # store for prediction
    return model

def predict_lgbm(model, X_v, X_tr, cat_feats):
    _, X_v_unified = unify_categories(X_tr, X_v, cat_feats)
    return model.predict(X_v_unified)

# XGBoost train/predict
def train_xgb_tuned(X_tr, y_tr, X_v, y_v, cat_feats):
    X_tr_enc = encode_categoricals(X_tr, cat_feats, label_encoders)
    X_v_enc = encode_categoricals(X_v, cat_feats, label_encoders)
    dtrain = xgb.DMatrix(X_tr_enc, y_tr)
    dval = xgb.DMatrix(X_v_enc, y_v)
    model = xgb.train(best_xgb_params, dtrain, num_boost_round=1000,
                      evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
    return model

def predict_xgb(model, X_v, X_tr, cat_feats):
    X_v_enc = encode_categoricals(X_v, cat_feats, label_encoders)
    return model.predict(xgb.DMatrix(X_v_enc))

# CatBoost train/predict
def train_cb_tuned(X_tr, y_tr, X_v, y_v, cat_feats):
    X_tr, X_v = X_tr.copy(), X_v.copy()
    for col in cat_feats:
        X_tr[col] = X_tr[col].fillna('unknown').astype(str)
        X_v[col] = X_v[col].fillna('unknown').astype(str)
    cat_indices = [list(X_tr.columns).index(c) for c in cat_feats]
    model = CatBoostRegressor(
        iterations=1000, **best_cb_params,
        random_seed=SEED, verbose=0, early_stopping_rounds=50, cat_features=cat_indices)
    model.fit(X_tr, y_tr, eval_set=(X_v, y_v), verbose=0)
    return model

def predict_cb(model, X_v, X_tr, cat_feats):
    X_v = X_v.copy()
    for col in cat_feats:
        X_v[col] = X_v[col].fillna('unknown').astype(str)
    return model.predict(X_v)

# Collect OOF predictions
print('\n--- Collecting OOF predictions ---')

print('\nlgbm_li-gw-wp-op_v1:')
lgbm_scores, lgbm_time, oof_lgbm = run_cv_full(
    'lgbm', train_lgbm_tuned, predict_lgbm, air_visit, VAL_FOLDS,
    ALL_FEATURES, CATEGORICAL_FEATURES, collect_oof=True)

print('\nxgb_li-gw-wp-op_v1:')
xgb_scores, xgb_time, oof_xgb = run_cv_full(
    'xgb', train_xgb_tuned, predict_xgb, air_visit, VAL_FOLDS,
    ALL_FEATURES, CATEGORICAL_FEATURES, collect_oof=True)

print('\ncb_li-gw-wp-op_v1:')
cb_scores, cb_time, oof_cb = run_cv_full(
    'cb', train_cb_tuned, predict_cb, air_visit, VAL_FOLDS,
    ALL_FEATURES, CATEGORICAL_FEATURES, collect_oof=True)

---
## 5. [st] スタッキング（2段階アンサンブル）

### 構造
- **Level 1**: LightGBM / XGBoost / CatBoost（各Optuna最適化済み）
- **Level 2**: Ridge回帰（OOF予測のlog値を入力、overfitting防止のため線形モデル）

### なぜRidgeか
- Level 2が複雑すぎると過学習リスク → 線形モデルが安定
- L2正則化でモデル間の重みを自動調整
- 3入力しかないため、非線形モデルは不要

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5a. [st] OOF予測の結合 & スタッキングLevel 2
# ══════════════════════════════════════════════════════════════

# OOF予測を結合
oof_merged = oof_lgbm[['air_store_id', 'visit_date', 'fold', 'actual']].copy()
oof_merged['pred_log_lgbm'] = oof_lgbm['pred_log_lgbm']
oof_merged['pred_log_xgb'] = oof_xgb['pred_log_xgb']
oof_merged['pred_log_cb'] = oof_cb['pred_log_cb']

print(f'OOF merged: {oof_merged.shape}')
print(f'Correlation of OOF predictions (log space):')
print(oof_merged[['pred_log_lgbm', 'pred_log_xgb', 'pred_log_cb']].corr().round(3))

# ── Level 2: Ridge on OOF ──
# fold-wise CV for stacking to avoid leakage
stack_scores = []
meta_model = None

X_meta = oof_merged[['pred_log_lgbm', 'pred_log_xgb', 'pred_log_cb']].values
y_meta = np.log1p(oof_merged['actual'].values)

# Train Ridge on all OOF data (each sample was predicted out-of-fold by Level 1)
meta_model = Ridge(alpha=1.0)
meta_model.fit(X_meta, y_meta)

# Evaluate stacking CV (fold-wise)
for fold_idx in range(len(VAL_FOLDS)):
    mask = oof_merged['fold'] == fold_idx
    X_fold = X_meta[mask]
    y_fold_actual = oof_merged.loc[mask, 'actual'].values
    
    y_fold_pred = np.expm1(meta_model.predict(X_fold))
    y_fold_pred = np.maximum(y_fold_pred, 0)
    score = rmsle(y_fold_actual, y_fold_pred)
    stack_scores.append(score)
    print(f'  Fold {fold_idx+1}: {score:.4f}')

print(f'  Stack Mean: {np.mean(stack_scores):.4f} (±{np.std(stack_scores):.4f})')
print(f'\nRidge coefficients: lgbm={meta_model.coef_[0]:.3f}, xgb={meta_model.coef_[1]:.3f}, cb={meta_model.coef_[2]:.3f}')
print(f'Ridge intercept: {meta_model.intercept_:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5b. モデル比較サマリー & 可視化
# ══════════════════════════════════════════════════════════════

# 06_model_designの結果も含めた比較表
model_registry = {
    'baseline_dow_median': {'scores': baseline_scores, 'elements': '(none)', 'source': '06'},
    'lgbm_li-gw-wp-op_v1': {'scores': lgbm_scores, 'elements': 'li+gw+wp+op', 'source': '07'},
    'xgb_li-gw-wp-op_v1':  {'scores': xgb_scores,  'elements': 'li+gw+wp+op', 'source': '07'},
    'cb_li-gw-wp-op_v1':   {'scores': cb_scores,   'elements': 'li+gw+wp+op', 'source': '07'},
    'stack_li-gw-wp-op-st_v1': {'scores': stack_scores, 'elements': 'li+gw+wp+op+st', 'source': '07'},
}

print('=' * 80)
print('MODEL REGISTRY — Full Comparison')
print('=' * 80)
print(f'\n{"Model":<30s} {"Mean RMSLE":>11s} {"±Std":>8s} {"vs Base":>9s} {"Elements":>18s}')
print('-' * 80)
for name, info in sorted(model_registry.items(), key=lambda x: np.mean(x[1]['scores'])):
    m = np.mean(info['scores'])
    s = np.std(info['scores'])
    diff = mean_baseline - m
    print(f'{name:<30s} {m:>11.4f} {s:>8.4f} {diff:>+9.4f} {info["elements"]:>18s}')

# ── Visualization ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Fold-by-fold
fold_labels = ['F1\n(GW)', 'F2\n(夏)', 'F3\n(秋)', 'F4\n(年末)', 'F5\n(春)']
x = np.arange(len(fold_labels))
w = 0.15
models_to_plot = ['baseline_dow_median', 'lgbm_li-gw-wp-op_v1', 'xgb_li-gw-wp-op_v1',
                  'cb_li-gw-wp-op_v1', 'stack_li-gw-wp-op-st_v1']
colors = ['gray', '#4C72B0', '#DD8452', '#55A868', '#C44E52']

for i, name in enumerate(models_to_plot):
    short = name.split('_')[0] if '_' in name else name
    axes[0].bar(x + i * w, model_registry[name]['scores'], w, label=short, color=colors[i], alpha=0.8)
axes[0].set_xticks(x + w * 2)
axes[0].set_xticklabels(fold_labels)
axes[0].set_ylabel('RMSLE')
axes[0].set_title('Fold別 RMSLE')
axes[0].legend(fontsize=9)

# Mean comparison
names = [n for n, _ in sorted(model_registry.items(), key=lambda x: np.mean(x[1]['scores']))]
means = [np.mean(model_registry[n]['scores']) for n in names]
stds = [np.std(model_registry[n]['scores']) for n in names]
short_names = [n.replace('_li-gw-wp-op', '\nfull').replace('_v1', '').replace('_li-gw-wp-op-st', '\nfull+st') for n in names]

axes[1].barh(range(len(names)), means, xerr=stds, capsize=4, alpha=0.8,
             color=['#C44E52', '#4C72B0', '#55A868', '#DD8452', 'gray'][:len(names)])
axes[1].set_yticks(range(len(names)))
axes[1].set_yticklabels(short_names, fontsize=9)
axes[1].set_xlabel('Mean RMSLE (±std)')
axes[1].set_title('モデル別 平均RMSLE')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 6. 最終学習 → Submission生成

全学習データで再学習し、テスト予測を生成する。
スタッキングとシンプルな重み付きアンサンブルの2種類のSubmissionを作成。